In [13]:
from scenario_access import load_scenario

data=load_scenario(32,"",False)

********************************************************
Loading Scenario32 dataset from 


C:\Users\Sedat Cimen\deepsense_positon_non_linear\scenario_access.py:92: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  scenario_df.interpolate(method='linear', axis=0, limit_direction='both',inplace=True)


In [14]:
data.head()

,unit2_spd_over_grnd_kmph,unit2_altitude,unit2_geo_sep,unit2_mode_fix_type,unit2_pdop,unit2_hdop,unit2_vdop,unit2_interpolated_position,unit2_lat,unit2_lon,unit1_beam_index
0,5.820,356.810,-27.749,3.0,1.09,0.58,0.92,0.0,33.424312,-111.935136,2
1,5.841,356.812,-27.749,3.0,1.09,0.58,0.92,1.0,33.424312,-111.935134,2
2,5.862,356.814,-27.749,3.0,1.09,0.58,0.92,0.0,33.424312,-111.935133,2
3,6.066,356.814,-27.749,3.0,1.09,0.58,0.92,0.0,33.424312,-111.935133,2
4,6.351,356.807,-27.749,3.0,1.09,0.58,0.92,0.0,33.424312,-111.935131,4


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# --- GEOMETRİK HESAPLAMALAR ---
#
#def get_distance(lat1, lon1, lat2, lon2):
#    R = 6371000 # metre cinsinden dünya yarıçapı
#    phi1, phi2 = np.radians(lat1), np.radians(lat2)
#    dphi = np.radians(lat2 - lat1)
#    dlambda = np.radians(lon2 - lon1)
#    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
#    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
#
#def get_bearing(lat1, lon1, lat2, lon2):
#    dLon = np.radians(lon2 - lon1)
#    y = np.sin(dLon) * np.cos(np.radians(lat2))
#    x = np.cos(np.radians(lat1)) * np.sin(np.radians(lat2)) - \
#        np.sin(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.cos(dLon)
#    return np.degrees(np.arctan2(y, x))
#
## Sabit Baz İstasyonu 
#u1_lat, u1_lon = 33.4251, -111.9352 
#
## Öznitelikler
#data['dist_to_bs'] = get_distance(u1_lat, u1_lon, data['unit2_lat'], data['unit2_lon'])
#data['bearing'] = get_bearing(u1_lat, u1_lon, data['unit2_lat'], data['unit2_lon'])
# Mevcut sütunların üzerine farkları (vektörleri) ekle

#data['lat_diff'] = data['unit2_lat'].diff().fillna(0)
#data['lon_diff'] = data['unit2_lon'].diff().fillna(0)


########################lat lon farkları haversine mesafesi ile birleştirildi ve unit2_prev_distance yani önceki mesafe eklendi.
######################## bearing ve dist_to_bs yeni adı unit1_unit2_distance ile Data Access'e eklendi, daha sonraki committe silinecek


In [ ]:
# Kullanılacak özellik listesi 
features = [
    'unit2_lat', 'unit2_lon', 'unit2_spd_over_grnd_kmph', 
    'unit2_altitude', 'unit2_pdop', 'bearing',"unit1_unit2_distance",	"unit2_prev_distance"
]
#####################LSTM için Feature Selection methodu kullanılmalı, 
##################### Xgboost ve RF için feature selectiona gerek yok çünkü otomatik yapıyor.
#####################  tüm featureler verilebilir. Bu kısma ben çalışacağım.

##################### Skor Hesaplama için CV daha doğru olacaktır.


##################### Normalizasyon train test split'ten sonra yapılmalı
#scaler = MinMaxScaler()
#X_scaled = scaler.fit_transform(data[features])
#y_raw = data['unit1_beam_index'].values


#X_final, y_final = create_windows(X_scaled, y_raw, window_size=10)

# 
#X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.2, random_state=42)
##################### CV akışı şu şekilde yapılabilir.
window_size=10
def create_windows(X, y, window_size=10):
    X_win, y_win = [], []
    for i in range(len(X) - window_size):
        X_win.append(X[i : i + window_size])
        y_win.append(y[i + window_size])
    return np.array(X_win), np.array(y_win)

def generate_sequence_windows(data, window_size=10):
    """
    Generate sequence windows for all groups in the provided DataFrame, then concatenate all into X, y arrays.

    Args:
        data (pd.DataFrame): The input data containing the 'seq_index' column and 'unit1_beam_index'.
        window_size (int): The length of the sliding window.

    Returns:
        X_all (np.ndarray): All input windows concatenated from every group.
        y_all (np.ndarray): All target values concatenated from every group.
    """
    grouped = data.groupby('seq_index')
    X_list = []
    y_list = []
    for seq_idx, group in grouped:
        group = group.reset_index(drop=True)
        X = group.drop(columns=['unit1_beam_index']).to_numpy()
        y = group['unit1_beam_index'].to_numpy()
        X_win, y_win = create_windows(X, y, window_size)
        X_list.append(X_win)
        y_list.append(y_win)
    X_all = np.concatenate(X_list, axis=0)
    y_all = np.concatenate(y_list, axis=0)
    return X_all, y_all

from sklearn.preprocessing import StandardScaler
features = [
    'unit2_lat', 'unit2_lon', 'unit2_spd_over_grnd_kmph', 
    'unit2_altitude', 'unit2_pdop', 'bearing',"unit1_unit2_distance",	"unit2_prev_distance"
]
# 5 fold split on the grouped (seq-based) windows
n_folds = 5

from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=n_folds)
folds = []
for fold_num, (train_idx, test_idx) in enumerate(gkf.split(data, groups=data["seq_index"])):
    train=data.iloc[train_idx]
    test=data.iloc[test_idx]
    scaler = StandardScaler()
    train.loc[:,features] = scaler.fit_transform(train[features])
    test.loc[:,features] = scaler.transform(test[features])

    X_train, y_train = generate_sequence_windows(train, window_size)

    
    X_test, y_test = generate_sequence_windows(test, window_size)

 
    train_seq_indices = data.iloc[train_idx]['seq_index'].unique()
    test_seq_indices = data.iloc[test_idx]['seq_index'].unique()
    print(f"Fold {fold_num+1}:")
    print(f"  Train instances: {len(X_train)}")
    print(f"  Test instances: {len(X_test)}")
    print(f"  Train X shape: {X_train.shape}")
    print(f"  Test X shape: {X_test.shape}")
    print(f"  Train y shape: {y_train.shape}")
    print(f"  Test y shape: {y_test.shape}")
    print(f"  Unique seq_index in train: {train_seq_indices}")
    print(f"  Unique seq_index in test: {test_seq_indices}")
    ##################### Burdan sonra model eğitimi ve testi yapılabilir.



In [9]:
# 64 farklı hüzme 
num_classes = 64 

model = Sequential([
    LSTM(128, input_shape=(10, len(features)), return_sequences=True),
    BatchNormalization(),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=0.001), 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

# --- FINE-TUNE ARAÇLARI 
# EarlyStopping: 
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# ReduceLROnPlateau: 
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.00001)

# EĞİTİM
history = model.fit(
    X_train, y_train,
    epochs=100, # 
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

C:\Users\Sedat Cimen\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.1767 - loss: 3.1714 - val_accuracy: 0.1752 - val_loss: 3.1977 - learning_rate: 0.0010
Epoch 2/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.2543 - loss: 2.4961 - val_accuracy: 0.2109 - val_loss: 2.8753 - learning_rate: 0.0010
Epoch 3/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.2822 - loss: 2.3284 - val_accuracy: 0.2124 - val_loss: 2.5125 - learning_rate: 0.0010
Epoch 4/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.2903 - loss: 2.2827 - val_accuracy: 0.3504 - val_loss: 2.2607 - learning_rate: 0.0010
Epoch 5/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.3023 - loss: 2.1851 - val_accuracy: 0.3163 - val_loss: 2.2535 - learning_rate: 0.0010
Epoch 6/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.3124 - loss: 2.1396 - val_accuracy: 0.2992 - val_loss: 2.2423 - learning_rate: 0.0010
Epoch 7/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.3326 - loss: 2.0512 - 

In [10]:
import tensorflow as tf

# Modelin test seti üzerindeki olasılık tahminleri
y_pred_probs = model.predict(X_test)

# Top-1 Accuracy 
top1_acc = tf.keras.metrics.sparse_top_k_categorical_accuracy(y_test, y_pred_probs, k=1)
# Top-3 Accuracy
top3_acc = tf.keras.metrics.sparse_top_k_categorical_accuracy(y_test, y_pred_probs, k=3)
# Top-5 Accuracy 
top5_acc = tf.keras.metrics.sparse_top_k_categorical_accuracy(y_test, y_pred_probs, k=5)

print(f"Top-1 Doğruluk (Tam İsabet): %{np.mean(top1_acc)*100:.2f}")
print(f"Top-3 Doğruluk: %{np.mean(top3_acc)*100:.2f}")
print(f"Top-5 Doğruluk: %{np.mean(top5_acc)*100:.2f}")

21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step
Top-1 Doğruluk (Tam İsabet): %42.95
Top-3 Doğruluk: %76.12
Top-5 Doğruluk: %87.44


In [11]:
# XGBoost ve Random Forest

In [22]:
#RF yeniden
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import tensorflow as tf

# --- 1. VERİYİ RF FORMATINA DÖNÜŞTÜRME ---
def flatten_windows(X):
    return X.reshape(X.shape[0], -1)

X_train_rf = flatten_windows(X_train)
X_test_rf = flatten_windows(X_test)

# --- 2. MODELİ KURMA VE EĞİTME ---
rf_model = RandomForestClassifier(
    n_estimators=200,          
    max_depth=20,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"    

print("Random Forest Eğitimi Başlıyor...")
rf_model.fit(X_train_rf, y_train)

# --- 3. TAHMİN VE TOP-K ANALİZİ ---
y_pred_rf = rf_model.predict(X_test_rf)
y_probs_rf = rf_model.predict_proba(X_test_rf)

# Tensorflow 
y_probs_rf_tf = tf.cast(y_probs_rf, tf.float32)
y_test_tf = tf.cast(y_test, tf.int32)

# Top-1
rf_acc = accuracy_score(y_test, y_pred_rf)

# Top-k
top3_rf = tf.keras.metrics.sparse_top_k_categorical_accuracy(y_test_tf, y_probs_rf_tf, k=3)
top5_rf = tf.keras.metrics.sparse_top_k_categorical_accuracy(y_test_tf, y_probs_rf_tf, k=5)

print("-" * 40)
print("RANDOM FOREST SONUÇLARI:")
print(f"Top-1 Accuracy: %{rf_acc*100:.2f}")
print(f"Top-3 Accuracy: %{np.mean(top3_rf)*100:.2f}")
print(f"Top-5 Accuracy: %{np.mean(top5_rf)*100:.2f}")
print("-" * 40)

# --- 4. CLASSIFICATION REPORT ---
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

# --- 5. GERÇEK FEATURE IMPORTANCE (WINDOW TOPLAMLI) 
importances = rf_model.feature_importances_

window_size = X_train.shape[1]
feature_count = X_train.shape[2]

print("\nGerçek Feature Importance :")

for i in range(feature_count):
    importance_sum = 0
    for t in range(window_size):
        importance_sum += importances[t * feature_count + i]
    print(f"{features[i]}: {importance_sum:.4f}")

Random Forest Eğitimi Başlıyor...
----------------------------------------
RANDOM FOREST SONUÇLARI:
Top-1 Accuracy: %42.95
Top-3 Accuracy: %54.57
Top-5 Accuracy: %75.66
----------------------------------------

Classification Report:
              precision    recall  f1-score   support

           1       0.00      0.00      0.00         7
           2       0.24      0.40      0.30        10
           3       0.15      0.22      0.18         9
           4       0.29      0.20      0.24        10
           5       0.46      0.52      0.49        23
           6       0.40      0.15      0.22        13
           7       0.24      0.80      0.36         5
           8       0.00      0.00      0.00         4
           9       0.22      0.33      0.27        12
          10       0.41      0.41      0.41        17
          11       0.20      0.08      0.12        12
          12       0.29      0.21      0.24        19
          13       0.14      0.21      0.17        14
         

C:\Users\Sedat Cimen\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Sedat Cimen\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Sedat Cimen\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Sedat Cimen\anaconda3

In [23]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import tensorflow as tf

# 
#etiketleri 0 tabanlı yapma
min_label = np.min(y_train)

if min_label == 0:
    y_train_xgb_zero = y_train
    y_test_xgb_zero = y_test
else:
    y_train_xgb_zero = y_train - min_label
    y_test_xgb_zero = y_test - min_label

num_classes = len(np.unique(y_train_xgb_zero))

# --- 2. MODEL KURULUMU ---
xgb_model = xgb.XGBClassifier(
    n_estimators=200,           
    learning_rate=0.1,
    max_depth=6,
    objective='multi:softprob',
    num_class=num_classes,      
    random_state=42,
    eval_metric='mlogloss',
    tree_method='hist'          
)

print("XGBoost Eğitimi Başlıyor...")
xgb_model.fit(X_train_rf, y_train_xgb_zero)

#  3. TAHMİN VE TOP-K ANALİZİ 
y_pred_xgb_zero = xgb_model.predict(X_test_rf)
y_probs_xgb = xgb_model.predict_proba(X_test_rf)

# Tensorflow 
y_probs_xgb_tf = tf.cast(y_probs_xgb, tf.float32)
y_test_tf = tf.cast(y_test_xgb_zero, tf.int32)

#  4. METRİKLER 
xgb_acc = accuracy_score(y_test_xgb_zero, y_pred_xgb_zero)

top3_xgb = tf.keras.metrics.sparse_top_k_categorical_accuracy(
    y_test_tf, y_probs_xgb_tf, k=3
)

top5_xgb = tf.keras.metrics.sparse_top_k_categorical_accuracy(
    y_test_tf, y_probs_xgb_tf, k=5
)

print("-" * 40)
print("XGBOOST SONUÇLARI:")
print(f"Top-1 Accuracy: %{xgb_acc*100:.2f}")
print(f"Top-3 Accuracy: %{np.mean(top3_xgb)*100:.2f}")
print(f"Top-5 Accuracy: %{np.mean(top5_xgb)*100:.2f}")
print("-" * 40)

# --- 5. CLASSIFICATION REPORT ---
print("\nClassification Report:")
print(classification_report(y_test_xgb_zero, y_pred_xgb_zero))

XGBoost Eğitimi Başlıyor...
----------------------------------------
XGBOOST SONUÇLARI:
Top-1 Accuracy: %43.41
Top-3 Accuracy: %75.19
Top-5 Accuracy: %85.89
----------------------------------------

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         7
           1       0.31      0.40      0.35        10
           2       0.12      0.22      0.16         9
           3       0.12      0.10      0.11        10
           4       0.42      0.57      0.48        23
           5       0.25      0.08      0.12        13
           6       0.25      0.80      0.38         5
           7       0.00      0.00      0.00         4
           8       0.21      0.25      0.23        12
           9       0.33      0.41      0.37        17
          10       0.12      0.08      0.10        12
          11       0.29      0.26      0.28        19
          12       0.17      0.14      0.15        14
          13       0.

C:\Users\Sedat Cimen\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Sedat Cimen\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Sedat Cimen\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Sedat Cimen\anaconda3

In [25]:
np.bincount(y_train)  #Her beam classından kaç tane var

array([  0,  27,  52,  42,  41,  82,  36,  34,  23,  65,  66,  28,  63,
        67,  36, 225, 357,  39,  81, 158,  97, 105,  50,  10,  36,  93,
        31,  53,  45,  15,  16,  15,  52,  54,   6,  19,  35,  29,  38,
         9,   4,  17,  16,  14,  11,  13,  13,  15,   9,   3,  18,   2,
         9,  17,  22,  11,   4,  12,  17,   7,   5,  11], dtype=int64)